In [ ]:
# Installation automatique des dépendances requises dans le noyau Jupyter actuel
%pip install -r ../requirements.txt

# 🧠 Étape 5 : Modélisation (Machine Learning & Deep Learning) (Squelette Étudiant)

Cette étape correspond au cinquième chapitre du cours. L'objectif est d'implémenter d'une part un modèle de Machine Learning tabulaire (ex: RandomForest) et d'autre part un réseau de neurones convolutif (CNN) sous TensorFlow pour traiter des images ou signaux complexes.

### 1. Préparation de l'environnement

On importe les librairies nécessaires pour la modélisation.
On utilise **scikit-learn** pour le Machine Learning tabulaire et **MLPRegressor**
comme démonstration de réseau de neurones sur des données tabulaires.

In [28]:
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.linear_model import LinearRegression
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (mean_squared_error, r2_score,
                             mean_absolute_error, classification_report)

%matplotlib inline
print("Librairies de modélisation importées avec succès !")

Librairies de modélisation importées avec succès !


### 2. Modélisation Tabulaire (Machine Learning)

On entraîne deux modèles supervisés pour **prédire le MPG** (consommation en miles par gallon) :
- **Régression Linéaire** : modèle simple et interprétable, bon pour établir une baseline
- **Random Forest** : modèle ensembliste plus puissant, capable de capturer des relations non-linéaires

#### Pourquoi ces 2 modèles ?
La régression linéaire nous donne une référence de base (baseline). Si le Random Forest
fait significativement mieux, cela prouve que les relations entre variables ne sont pas
purement linéaires.

#### Chargement et préparation des données

In [29]:
# Chargement des données nettoyées
df = pd.read_csv('../data/processed/cleaned_data_sample.csv')

# Feature Engineering
df['power_to_weight'] = df['horsepower'] / df['weight']

# Définition des features et de la cible
X = df.drop(columns=['mpg'])
y = df['mpg']

# Split train/test (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Taille du jeu d'entraînement : {X_train.shape}")
print(f"Taille du jeu de test        : {X_test.shape}")
print(f"\nFeatures utilisées : {list(X.columns)}")

Taille du jeu d'entraînement : (318, 8)
Taille du jeu de test        : (80, 8)

Features utilisées : ['cylinders', 'displacement', 'horsepower', 'weight', 'acceleration', 'model_year', 'origin', 'power_to_weight']


#### Entraînement des modèles

On entraîne les 2 modèles sur le jeu d'entraînement.
Le **random_state=42** garantit la reproductibilité des résultats.

In [30]:
# Modèle 1 : Régression Linéaire (baseline)
lr_model = LinearRegression()
lr_model.fit(X_train, y_train)
print("✅ Régression Linéaire entraînée !")

# Modèle 2 : Random Forest
rf_model = RandomForestRegressor(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)
print("✅ Random Forest entraîné !")

# Prédictions sur le jeu de test
y_pred_lr = lr_model.predict(X_test)
y_pred_rf = rf_model.predict(X_test)

# Aperçu des premières prédictions vs valeurs réelles
comparison = pd.DataFrame({
    'Valeur Réelle': y_test.values[:10],
    'Prédiction LR': y_pred_lr[:10].round(2),
    'Prédiction RF': y_pred_rf[:10].round(2)
})
print("\nAperçu des 10 premières prédictions :")
print(comparison.to_string(index=False))

✅ Régression Linéaire entraînée !
✅ Random Forest entraîné !

Aperçu des 10 premières prédictions :
 Valeur Réelle  Prédiction LR  Prédiction RF
          33.0          32.86          30.60
          28.0          29.96          29.32
          19.0          21.06          19.93
          13.0          17.06          14.75
          14.0          12.33          14.36
          27.0          24.69          25.65
          24.0          28.10          26.08
          13.0          11.43          11.96
          17.0          15.75          17.95
          21.0          22.02          19.38


#### Importance des variables (Feature Importance)

Le Random Forest calcule l'importance de chaque variable dans la prédiction.
Plus le score est élevé, plus la variable contribue à expliquer le MPG.
Cela nous permet de savoir **quelles caractéristiques techniques influencent
le plus la consommation de carburant**.

In [3]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import os
from sklearn.ensemble import RandomForestRegressor

# 1. L'ANTIDOTE À L'AMNÉSIE : On recharge les données et on entraîne le modèle rapidement
df = pd.read_csv('../data/processed/cleaned_data_sample.csv')
X = df.drop(columns=['mpg', 'mpg_category'], errors='ignore')
y = df['mpg']

rf_model = RandomForestRegressor(n_estimators=100, random_state=42)
rf_model.fit(X, y)

# 2. Récupération et tri des importances (Maintenant X et rf_model existent !)
feature_names = X.columns.tolist()
importances = rf_model.feature_importances_

# Tri décroissant
indices = np.argsort(importances)[::-1]
sorted_features = [feature_names[i] for i in indices]
sorted_importances = importances[indices]

# 3. Création du graphique à barres interactif
fig = go.Figure(data=[go.Bar(
    x=sorted_features,
    y=sorted_importances,
    marker_color='#3b82f6',               
    text=np.round(sorted_importances, 4), 
    textposition='auto',                  
    hovertemplate="<b>%{x}</b><br>Importance : %{y:.4f}<extra></extra>"
)])

# 4. Mise en page globale
fig.update_layout(
    title_text="<b>Importance des variables — Random Forest (Explainable AI)</b>",
    title_font_size=16,
    xaxis_title="Variables (Caractéristiques du véhicule)",
    yaxis_title="Score d'importance",
    template="plotly_white",               
    height=600,
    xaxis_tickangle=-45                   
)

# 5. Sauvegarde en format Web Interactif (.html)
output_dir = '../data/processed'
os.makedirs(output_dir, exist_ok=True)
html_path = os.path.join(output_dir, 'feature_importance.html')
fig.write_html(html_path)

print(f"✅ Graphique interactif sauvegardé : {html_path}")

# Affichage interactif dans le Notebook
fig.show()

print("\n🏆 Top 3 variables les plus importantes :")
for i in range(3):
    print(f"  {sorted_features[i]}: {sorted_importances[i]:.4f}")

✅ Graphique interactif sauvegardé : ../data/processed\feature_importance.html



🏆 Top 3 variables les plus importantes :
  displacement: 0.3032
  weight: 0.2235
  cylinders: 0.2060


In [32]:
# Calcul des métriques
y_pred = y_pred_rf

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print(f"Erreur Absolue Moyenne (MAE)           : {mae:.2f} MPG")
print(f"Racine de l'Erreur Quadratique (RMSE)  : {rmse:.2f} MPG")
print(f"Score R² (Précision globale)           : {r2:.2f}")

Erreur Absolue Moyenne (MAE)           : 1.61 MPG
Racine de l'Erreur Quadratique (RMSE)  : 2.15 MPG
Score R² (Précision globale)           : 0.91


#### Interprétation des résultats

| Métrique | Valeur | Signification |
|---|---|---|
| **MAE** | 1.61 MPG | En moyenne, notre modèle se trompe de 1.61 miles par gallon |
| **RMSE** | 2.15 MPG | Les grosses erreurs sont rares — le modèle est stable |
| **R²** | 0.91 | Le modèle explique **91% de la variance** du MPG |

**Ce que ça veut dire en clair :**

- Un R² de 0.91 est **excellent** — le modèle capture très bien les patterns
- Une MAE de 1.61 MPG signifie que si une voiture fait 25 MPG réellement,
  notre modèle prédit entre 23.4 et 26.6 MPG en moyenne
- Le Random Forest fait clairement mieux qu'une simple régression linéaire,
  ce qui confirme que les relations entre les variables **ne sont pas linéaires**

#### Classification — Prédiction de la catégorie de consommation

En plus de prédire la valeur exacte du MPG, on peut aussi **classer** les voitures
en 3 catégories : consommation faible, moyenne ou élevée.
On utilise un **Random Forest Classifier** pour cette tâche.

In [33]:
# Création de la variable cible catégorielle
df['mpg_category'] = pd.qcut(df['mpg'], q=3, labels=[0, 1, 2])

X_clf = df.drop(columns=['mpg', 'mpg_category'])
y_clf = df['mpg_category'].astype(int)

X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(
    X_clf, y_clf, test_size=0.2, random_state=42
)

# Entraînement
clf_model = RandomForestClassifier(n_estimators=100, random_state=42)
clf_model.fit(X_train_c, y_train_c)
y_pred_c = clf_model.predict(X_test_c)

print("✅ Classificateur entraîné !")
print("\nRapport de classification :")
print(classification_report(y_test_c, y_pred_c,
      target_names=['Faible', 'Moyenne', 'Élevée']))

✅ Classificateur entraîné !

Rapport de classification :
              precision    recall  f1-score   support

      Faible       0.93      0.89      0.91        28
     Moyenne       0.81      0.86      0.83        29
      Élevée       0.91      0.87      0.89        23

    accuracy                           0.88        80
   macro avg       0.88      0.87      0.88        80
weighted avg       0.88      0.88      0.88        80



#### Interprétation des résultats — Classification

Les résultats du rapport de classification montrent que le modèle obtient
une **accuracy globale de 88%** sur le jeu de test.

| Catégorie | Précision | Recall | F1-Score | Support |
|---|---|---|---|---|
| Faible (0) | 0.93 | 0.89 | 0.91 | 28 |
| Moyenne (1) | 0.81 | 0.86 | 0.83 | 29 |
| Élevée (2) | 0.91 | 0.87 | 0.89 | 23 |
| **Global** | **0.88** | **0.88** | **0.88** | **80** |

**Ce que ça veut dire en clair :**

- **Accuracy de 88%** → le modèle classe correctement 88 voitures sur 100
- **Catégorie "Faible" mieux prédite** (F1=0.91) → les voitures gourmandes ont
  des caractéristiques très distinctives : gros moteurs, lourdes, beaucoup de cylindres
- **Catégorie "Moyenne" plus difficile** (F1=0.83) → ces voitures sont entre
  les deux extrêmes et peuvent ressembler aux deux autres catégories
- **Catégorie "Élevée" très bien prédite** (F1=0.89) → les voitures économiques
  sont aussi bien identifiées grâce à leur faible poids et petits moteurs
- **Conclusion** → le poids, la cylindrée et la puissance sont de très bons
  indicateurs de la catégorie de consommation d'une voiture

### 3. Modélisation Vision / Deep Learning (CNN & TensorFlow)

Notre dataset Automobile contient des chiffres dans un tableau — pas des images.
Un CNN est fait pour analyser des photos, donc il ne s'applique pas ici.

À la place, on utilise un **MLP (réseau de neurones)** qui fonctionne très bien
sur des données comme les nôtres. Il apprend à trouver des relations complexes
entre les variables pour prédire le MPG.

In [6]:
import os
import pandas as pd
import plotly.graph_objects as go
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import r2_score

# 1. L'ANTIDOTE : Chargement et découpage des données
df = pd.read_csv('../data/processed/cleaned_data_sample.csv')
X = df.drop(columns=['mpg', 'mpg_category'], errors='ignore')
y = df['mpg']

# On recrée X_train, X_test, y_train, y_test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 2. Normalisation - obligatoire pour les réseaux de neurones
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 3. Entraînement du Réseau de neurones (MLP)
mlp_model = MLPRegressor(
    hidden_layer_sizes=(64, 32),
    activation='relu',
    max_iter=500,
    random_state=42
)
mlp_model.fit(X_train_scaled, y_train)
y_pred_mlp = mlp_model.predict(X_test_scaled)

print("✅ Réseau de neurones entraîné !")
print(f"Nombre de couches  : {mlp_model.n_layers_}")
print(f"Nombre d'itérations : {mlp_model.n_iter_}")
print(f"R² MLP             : {r2_score(y_test, y_pred_mlp):.3f}")

# 4. Visualisation de la Courbe d'apprentissage avec PLOTLY
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=list(range(1, len(mlp_model.loss_curve_) + 1)),
    y=mlp_model.loss_curve_,
    mode='lines',
    name='Erreur (Loss)',
    line=dict(color='#3b82f6', width=3) 
))

fig.update_layout(
    title_text="<b>Courbe d'apprentissage — Réseau de Neurones (MLP)</b>",
    title_font_size=16,
    xaxis_title="Itérations (Époques)",
    yaxis_title="Erreur (Loss)",
    template="plotly_white",             
    height=500,
    hovermode="x unified"               
)

# 5. Sauvegarde en format Web Interactif (.html)
output_dir = '../data/processed'
os.makedirs(output_dir, exist_ok=True)
html_path = os.path.join(output_dir, 'mlp_loss_curve.html')
fig.write_html(html_path)

print(f"✅ Graphique interactif sauvegardé : {html_path}")

# Affichage interactif
fig.show()

✅ Réseau de neurones entraîné !
Nombre de couches  : 4
Nombre d'itérations : 500
R² MLP             : 0.883
✅ Graphique interactif sauvegardé : ../data/processed\mlp_loss_curve.html


c:\Users\tabbe\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(


### 4. Sauvegarde du modèle

On exporte le modèle Random Forest entraîné ainsi que le scaler vers le dossier `models/`
à la racine du projet. Ces fichiers seront chargés par l'API FastAPI du dashboard
pour effectuer des prédictions en temps réel sans avoir à ré-entraîner le modèle.

- **`model.pkl`** : le RandomForestRegressor entraîné (prédit le MPG)
- **`scaler.pkl`** : le StandardScaler ajusté sur X_train (normalise les entrées du MLP)

In [ ]:
import joblib
import os

# Créer le dossier models/ à la racine du projet
models_dir = os.path.join('..', 'models')
os.makedirs(models_dir, exist_ok=True)

# Sauvegarder le modèle Random Forest
model_path = os.path.join(models_dir, 'model.pkl')
joblib.dump(rf_model, model_path)
print(f' Modèle sauvegardé : {os.path.abspath(model_path)}')

# Sauvegarder le scaler
scaler_path = os.path.join(models_dir, 'scaler.pkl')
joblib.dump(scaler, scaler_path)
print(f' Scaler sauvegardé : {os.path.abspath(scaler_path)}')

# Afficher les features attendues par le modèle
feature_names = X.columns.tolist()
print(f'
 Features attendues par le modèle ({len(feature_names)}) :')
for i, name in enumerate(feature_names):
    print(f'  {i+1}. {name}')

print('
 Export terminé ! Les fichiers sont prêts pour l'API.')